# Journal Overlap & APC Exposure Analysis
## NLM Catalog · PMC · NIH-Funded Journals · Northwestern TA · Northwestern Publications

**Purpose:** Compare four journal lists using ISSN as the primary match key, classify each journal's open-access and APC cost profile, and estimate Northwestern's annual APC exposure for NIH-funded publications.

**Intended audience:** Library leadership, research administration, scholarly communications policy.

**Data sources:**

| Variable | Description | Scope |
|---|---|---|
| NLM Catalog | MEDLINE-indexed journals via NCBI E-utilities | Current index (`currentlyindexed[All]`) |
| PMC Journal List | Journals with PMC deposit agreements | Downloaded from NCBI CDN |
| NIH-Funded Journals | Top 2,228 journals by NIH paper volume | Jan–Jul 2025 only (7 months) |
| Northwestern TA — Wiley | Journals in Northwestern Wiley transformative agreement | APC waived |
| Northwestern TA — Springer Nature | Journals in Northwestern BTAA Springer Nature TA | APC waived |
| Northwestern Publications | NU NIH-funded papers from NIH Reporter | FY2020–present |

---
**Last revised:** See Configuration cell for run date.  
**Analyst:** Galter Health Sciences Library — Metrics and Impact Core


## ⚠️ Assumptions and Known Limitations

This section must be read before interpreting any outputs. These limitations are structural — they cannot be fully resolved without additional data — but they are quantified where possible in the Validation section.

### Matching methodology
- **ISSN is the sole match key.** Journals are matched across sources using normalized ISSN (print, electronic, and linking where available). Title-based matching is not performed. Journals with inconsistent ISSNs across sources will be missed or misclassified.
- **TA files contain eISSN only.** Both TA files provide only electronic ISSN. Journals with print-only ISSNs in the NLM Catalog may be falsely classified as not TA-covered.
- **ISSN instability.** ISSNs can change with journal title changes. The NLM linking ISSN partially mitigates this but does not resolve all historical cases.

### Source scope
- **NLM Catalog (`currentlyindexed[All]`)** returns only currently MEDLINE-indexed journals (~5,200). Non-MEDLINE biomedical journals are not present. This scope changes over time; a cache timestamp is printed at runtime.
- **NIH file covers Jan–Jul 2025 only (7 months),** not a full year. Journal rankings and publication counts reflect this partial-year window.
- **NIH file covers only the top 2,228 journals** by NIH paper volume. NU publications in journals outside this list have no APC pricing data available and contribute $0 to all financial estimates. The coverage gap is quantified in the Validation section.
- **NIH Reporter completeness.** The Northwestern publications file is sourced from NIH Reporter, which has known reporting lags and incomplete coverage of recent publications. Affiliation matching is imperfect.

### Financial estimates
- **APC estimates are hypothetical planning figures, not historical expenditure.** Formula: `(annual_average_NU_pub_count) × (2025_list_price_APC)`. This assumes every paper required a paid APC at the 2025 list price, which is not true in all cases.
- **List prices only.** Institutional discounts, member pricing, and individual waivers are not reflected.
- **APC liability classification assumes OA compliance intent.** For hybrid journals with immediate PMC release, we assume the PMC deposit satisfies NIH public access requirements without an APC payment. Authors who voluntarily pay hybrid OA APCs in these journals are not captured.
- **Journals not in the NIH 2,228 list have unknown APC prices.** Their liability category may be classifiable (if OA status is known from another source) but their dollar exposure cannot be estimated.

### What this analysis supports and does not support
| Supports | Does NOT support |
|---|---|
| Identifying which journals NU publishes in most actively | Calculating actual APC expenditure |
| Flagging TA coverage and potential savings | Proving TA agreements are being used |
| Estimating annual APC exposure order-of-magnitude | Precise budget forecasting |
| Comparing OA/hybrid/diamond distribution | Determining author-level payment decisions |


## 1. Imports

In [2]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import time
import pickle
import re
import io
from pathlib import Path
from datetime import datetime

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── NCBI credentials ──────────────────────────────────────────────────────────
from config import ENTREZ_EMAIL, ENTREZ_API_KEY

print("Imports OK")
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


Imports OK
Run date: 2026-04-01 16:56


## 2. Configuration

In [4]:

# ── NLM Catalog query ─────────────────────────────────────────────────────────
# "currentlyindexed[All]"  -> journals currently indexed in MEDLINE (~5,200)
# "ncbijournals[All]"      -> broader NLM journal collection (~60k)
# NOTE: Changing this requires deleting cache/nlm_journals.pkl to force a fresh fetch.
NLM_QUERY = "currentlyindexed[All]"

# ── External file paths ───────────────────────────────────────────────────────
PMC_CSV_URL = "https://cdn.ncbi.nlm.nih.gov/pmc/home/jlist.csv"
NIH_FILE    = Path("data/NIH2025_2228TopJournals.csv")
WILEY_FILE  = Path("data/2025_08-25 Wiley Hybrid and OA Journals.csv")
SN_FILE     = Path("data/2025_BTAA_Springer_Nature_hybrid_journals.csv")
NU_FILE     = Path("data/2025_09-05 Northwestern Pubs from NIH Reporter 2020 to present_All FY.csv")

# ── E-utilities ───────────────────────────────────────────────────────────────
EUTILS     = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
BATCH_SIZE = 500
SLEEP_SEC  = 0.11 if ENTREZ_API_KEY else 0.34

# ── Directories ───────────────────────────────────────────────────────────────
CACHE_DIR  = Path("cache")
OUTPUT_DIR = Path("output")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

NLM_CACHE = CACHE_DIR / "nlm_journals.pkl"
PMC_CACHE = CACHE_DIR / "pmc_journals.pkl"

# ── Verify local files exist ───────────────────────────────────────────────────
print("File existence check:")
for label, path in [("NIH", NIH_FILE), ("Wiley TA", WILEY_FILE),
                    ("SN TA", SN_FILE), ("NU pubs", NU_FILE)]:
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {label}: {path}")

print(f"\nNLM query   : {NLM_QUERY}")
print(f"API key     : {'set' if ENTREZ_API_KEY else 'not set (3 req/sec)'}")


File existence check:
  [OK] NIH: data\NIH2025_2228TopJournals.csv
  [OK] Wiley TA: data\2025_08-25 Wiley Hybrid and OA Journals.csv
  [OK] SN TA: data\2025_BTAA_Springer_Nature_hybrid_journals.csv
  [OK] NU pubs: data\2025_09-05 Northwestern Pubs from NIH Reporter 2020 to present_All FY.csv

NLM query   : currentlyindexed[All]
API key     : set


## 3. Helper Functions

In [5]:
def normalize_issn(raw):
    """
    Normalize an ISSN string to XXXX-XXXX format.
    Returns None if the input cannot be parsed as a valid 8-digit ISSN.
    """
    if not raw or not isinstance(raw, str):
        return None
    digits = re.sub(r'[^0-9Xx]', '', raw)
    if len(digits) == 8:
        return f"{digits[:4]}-{digits[4:]}".upper()
    return None


def any_issn_in_set(row, issn_cols, target_set):
    """
    Return True if any normalized ISSN from the given columns of a row
    appears in target_set. Used for membership classification.
    """
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in target_set:
            return True
    return False


def read_csv_robust(filepath, sep=None, **kwargs):
    """
    Read a CSV file trying multiple encodings in sequence.
    Returns (DataFrame, encoding_used). Raises ValueError if all fail.
    """
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            df = pd.read_csv(filepath, sep=sep, engine='python',
                             dtype=str, encoding=enc, **kwargs)
            print(f"  Loaded {filepath.name}: {len(df):,} rows | encoding={enc}")
            return df, enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {filepath} with utf-8-sig, cp1252, or latin-1")


def entrez_params(extras=None):
    """Return base E-utilities parameter dict merged with any extras."""
    p = {"email": ENTREZ_EMAIL}
    if ENTREZ_API_KEY:
        p["api_key"] = ENTREZ_API_KEY
    if extras:
        p.update(extras)
    return p


print("Helper functions defined.")


Helper functions defined.


## 4. Load Source A: NIH-Funded Journals (Top 2,228)

**Source:** Dimensions data as of 2025-07-24.  
**Scope:** Top 2,228 journals publishing NIH-funded papers, January–July 2025 only.  
**Limitation:** Partial-year data; H2-heavy journals may be underrepresented.


In [6]:
def load_nih_journals(filepath):
    """
    Load the NIH-funded journals file (tab or comma delimited).
    Extracts and normalizes up to 4 ISSN columns.
    Returns a cleaned DataFrame.
    """
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")

    # ── Normalize column names to snake_case ──────────────────────────────────
    orig_cols = list(df.columns)
    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    # ── Map to internal names using pattern matching ───────────────────────────
    rename_map = {
        next((c for c in df.columns if 'order' in c), None)                                      : 'nih_order',
        next((c for c in df.columns if c == 'journal'), None)                                    : 'nih_journal',
        next((c for c in df.columns if 'publisher' in c and 'type' not in c), None)              : 'nih_publisher',
        next((c for c in df.columns if 'publications' in c and 'nih' in c), None)                : 'nih_pub_count',
        next((c for c in df.columns if 'oa_status' in c or ('oa' in c and 'status' in c)), None) : 'nih_oa_status',
        next((c for c in df.columns if 'publisher_type' in c), None)                             : 'nih_publisher_type',
        next((c for c in df.columns if 'apc_2025' in c or ('apc' in c and '2025' in c)), None)  : 'nih_apc_2025_usd',
        next((c for c in df.columns if 'apc_category' in c), None)                              : 'nih_apc_category',
    }
    rename_map = {k: v for k, v in rename_map.items() if k is not None}
    df = df.rename(columns=rename_map)

    # ── Extract ISSN columns by original name (case-insensitive) ─────────────
    # The file has ISSN1–ISSN4 as original column names.
    # We search original columns because the snake_case transform renames them.
    orig_lower = [c.strip().lower() for c in orig_cols]
    for i in range(1, 5):
        target = f'issn{i}'
        if target in orig_lower:
            original_col = orig_cols[orig_lower.index(target)]
            # After rename, it's already been transformed to snake_case
            snake_col = re.sub(r'[^a-z0-9]+', '_', original_col.strip().lower()).strip('_')
            if snake_col in df.columns:
                df[f'nih_issn{i}'] = df[snake_col].apply(normalize_issn)
            else:
                df[f'nih_issn{i}'] = None
        else:
            df[f'nih_issn{i}'] = None

    # ── Numeric coercion ───────────────────────────────────────────────────────
    for col in ['nih_pub_count', 'nih_apc_2025_usd', 'nih_order']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


nih_df = load_nih_journals(NIH_FILE)

# ── Validate ISSN extraction ───────────────────────────────────────────────────
issn_cols_check = ['nih_issn1', 'nih_issn2', 'nih_issn3', 'nih_issn4']
issn_counts = {c: nih_df[c].notna().sum() for c in issn_cols_check if c in nih_df.columns}
print(f"\nISSN column population:")
for col, n in issn_counts.items():
    print(f"  {col}: {n:,} non-null values")

rows_with_any_issn = nih_df[[c for c in issn_cols_check if c in nih_df.columns]].notna().any(axis=1).sum()
print(f"  Rows with at least one ISSN: {rows_with_any_issn:,} / {len(nih_df):,}")

if rows_with_any_issn == 0:
    print("\n  WARNING: No ISSNs extracted. Check column names above and update ISSN extraction logic.")

# ── Build NIH ISSN lookup ─────────────────────────────────────────────────────
# Maps every normalized ISSN -> dict of NIH metadata for that journal.
# Last-write-wins if ISSNs overlap across rows (rare, logged below).

NIH_META_COLS = ['nih_order', 'nih_journal', 'nih_publisher', 'nih_pub_count',
                 'nih_oa_status', 'nih_publisher_type', 'nih_apc_2025_usd', 'nih_apc_category']
NIH_META_COLS = [c for c in NIH_META_COLS if c in nih_df.columns]

nih_lookup = {}
duplicate_issns = []
for _, row in nih_df.iterrows():
    entry = {c: row.get(c) for c in NIH_META_COLS}
    for col in ['nih_issn1', 'nih_issn2', 'nih_issn3', 'nih_issn4']:
        v = row.get(col)
        if pd.notna(v) and v:
            if v in nih_lookup:
                duplicate_issns.append(v)
            nih_lookup[v] = entry

nih_issns = set(nih_lookup.keys())
print(f"\nNIH lookup: {len(nih_issns):,} unique ISSNs from {len(nih_df):,} journals")
if duplicate_issns:
    print(f"  Note: {len(set(duplicate_issns))} ISSNs appeared in multiple rows (last-write-wins)")
NIH_EMPTY = {c: None for c in NIH_META_COLS}


  Loaded NIH2025_2228TopJournals.csv: 2,228 rows | encoding=cp1252
  Columns: ['Order (# NIH papers 01-07/2025)', 'Journal', 'ISSN1', 'ISSN2', 'ISSN3', 'ISSN4', 'Publisher', 'Publications acknowledging NIH funding (01-07/2025)', 'Journal OA status', 'Publisher type', 'APC 2025 (USD)', 'APC category', 'Total APCs (based on 01-07/2025)', 'APCs covered by $2k cap', '% covered by $2k cap', 'APCs not covered by $2k cap', 'APCs covered by $3k cap', '% covered by $3k cap', 'APCs not covered by $3k cap', 'APCs covered by $6 cap', '% covered by $6 cap', 'APCs not covered by $6 cap']

ISSN column population:
  nih_issn1: 2,228 non-null values
  nih_issn2: 1,623 non-null values
  nih_issn3: 33 non-null values
  nih_issn4: 3 non-null values
  Rows with at least one ISSN: 2,228 / 2,228

NIH lookup: 3,860 unique ISSNs from 2,228 journals
  Note: 27 ISSNs appeared in multiple rows (last-write-wins)


## 5. Load Source B: Northwestern NIH-Funded Publications

**Source:** NIH Reporter export, all fiscal years 2020–present.  
**Unit:** Publication-level (one row per paper). Aggregated to journal level by ISSN.  
**Known limitation:** NIH Reporter has reporting lags and incomplete affiliation matching.


In [7]:
def load_nu_journals(filepath):
    """
    Load NU publication-level data and aggregate to journal level.
    Returns (aggregated_df, n_years, year_min, year_max).
    """
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")

    # Find key columns by pattern
    issn_col  = next((c for c in df.columns if c.strip().upper() == 'ISSN'), None)
    grant_col = next((c for c in df.columns if 'core project' in c.lower()), None)
    year_col  = next((c for c in df.columns if 'pub year' in c.lower()), None)

    if not issn_col:
        raise ValueError(f"Could not find ISSN column in {filepath}. Columns: {list(df.columns)}")

    print(f"  ISSN column   : {issn_col}")
    print(f"  Grant column  : {grant_col}")
    print(f"  Year column   : {year_col}")

    # Normalize ISSN
    df['issn_norm'] = df[issn_col].apply(normalize_issn)

    # Drop rows with no usable ISSN
    before = len(df)
    df = df[df['issn_norm'].notna()].copy()
    print(f"  Rows with valid ISSN: {len(df):,} (dropped {before - len(df):,} no-ISSN rows)")

    # Extract year range for annual average calculation
    if year_col:
        years = pd.to_numeric(df[year_col], errors='coerce').dropna().unique()
        if len(years) > 0:
            n_years = len(years)
            year_min, year_max = int(years.min()), int(years.max())
        else:
            print("  WARNING: Year column present but no valid year values found. n_years=1.")
            n_years, year_min, year_max = 1, None, None
    else:
        print("  WARNING: Pub Year column not found. Annual average will equal multi-year total.")
        n_years, year_min, year_max = 1, None, None

    print(f"  Publication years: {year_min}–{year_max} ({n_years} year(s))")

    # Aggregate to journal level
    agg_dict = {'nu_pub_count': ('issn_norm', 'count')}
    if grant_col:
        agg_dict['nu_unique_grants'] = (grant_col, 'nunique')

    grouped = df.groupby('issn_norm').agg(**agg_dict).reset_index()
    print(f"  Unique journals (by ISSN): {len(grouped):,}")

    return grouped, n_years, year_min, year_max


nu_agg, nu_years, nu_year_min, nu_year_max = load_nu_journals(NU_FILE)

# Build NU ISSN lookup: issn -> {nu_pub_count, nu_unique_grants}
NU_META_COLS = [c for c in ['nu_pub_count', 'nu_unique_grants'] if c in nu_agg.columns]
nu_lookup = {
    row['issn_norm']: {c: row.get(c) for c in NU_META_COLS}
    for _, row in nu_agg.iterrows()
}
nu_issns = set(nu_lookup.keys())
NU_EMPTY = {c: None for c in NU_META_COLS}

print(f"\nNU lookup: {len(nu_issns):,} unique journals by ISSN")
print(f"Total NU-NIH publications in dataset: {nu_agg['nu_pub_count'].sum():,.0f}")


  Loaded 2025_09-05 Northwestern Pubs from NIH Reporter 2020 to present_All FY.csv: 27,474 rows | encoding=utf-8-sig
  Columns: ['Core Project Number', 'Affiliation', 'Pub Year', 'Authors', 'Country', 'ISSN', 'Journal Issue', 'Journal (Link to PubMed abstract)', 'Journal Title ABBR', 'Journal Volume', 'Language', 'Page Number', 'PMC ID', 'PMID', 'PUB Date', 'Title (Link to full-text in PubMed Central)', 'Related Publications in PubMed', 'Related Publications in Google Scholar', 'Articles Citing from PubMed Central', 'Articles Citing from Google Scholar', 'Relative Citation Ratio']
  ISSN column   : ISSN
  Grant column  : Core Project Number
  Year column   : Pub Year
  Rows with valid ISSN: 26,913 (dropped 561 no-ISSN rows)
  Publication years: 2020–2025 (6 year(s))
  Unique journals (by ISSN): 2,522

NU lookup: 2,522 unique journals by ISSN
Total NU-NIH publications in dataset: 26,913


## 6. Load Source C: Northwestern Transformative Agreement Journal Lists

**Sources:** Wiley TA journal list (eISSN + journal type); Springer Nature BTAA journal list (eISSN + publishing model + OA license).  
**Limitation:** Both files contain eISSN only. Journals with print-only ISSNs will not match.


In [8]:
def load_wiley_ta(filepath):
    """
    Load Wiley TA file.
    Expected columns: Journal Title, Online ISSN, Type
    """
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")

    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    rename_map = {}
    for c in df.columns:
        if re.search(r'title', c):            rename_map[c] = 'ta_title'
        elif re.search(r'issn', c):           rename_map[c] = 'ta_issn_raw'
        elif re.search(r'type|model', c):     rename_map[c] = 'ta_publishing_model'
    df = df.rename(columns=rename_map)

    if 'ta_issn_raw' not in df.columns:
        raise ValueError(f"Could not find ISSN column in Wiley file. Columns: {list(df.columns)}")

    df['ta_issn'] = df['ta_issn_raw'].apply(normalize_issn)
    df['ta_agreement'] = 'Wiley'

    before = len(df)
    df = df[df['ta_issn'].notna()].copy()
    print(f"  Rows with valid ISSN: {len(df):,} (dropped {before - len(df):,})")
    return df[['ta_title', 'ta_issn', 'ta_publishing_model', 'ta_agreement']].copy()


def load_sn_ta(filepath):
    """
    Load Springer Nature BTAA TA file.
    Expected columns: Journal Title, eISSN, Publishing Model, OA License
    """
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")

    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    rename_map = {}
    for c in df.columns:
        # 'title' but not 'imprint'
        if re.search(r'title', c) and 'imprint' not in c:  rename_map[c] = 'ta_title'
        elif re.search(r'^e_?issn$|eissn', c):             rename_map[c] = 'ta_issn_raw'
        elif re.search(r'publishing_model|model', c):       rename_map[c] = 'ta_publishing_model'
        elif re.search(r'oa_license|license', c):           rename_map[c] = 'ta_oa_license'
    df = df.rename(columns=rename_map)

    if 'ta_issn_raw' not in df.columns:
        raise ValueError(f"Could not find eISSN column in SN file. Columns: {list(df.columns)}")

    df['ta_issn'] = df['ta_issn_raw'].apply(normalize_issn)
    df['ta_agreement'] = 'Springer Nature'

    before = len(df)
    df = df[df['ta_issn'].notna()].copy()
    print(f"  Rows with valid ISSN: {len(df):,} (dropped {before - len(df):,})")

    keep = ['ta_title', 'ta_issn', 'ta_publishing_model', 'ta_agreement']
    if 'ta_oa_license' in df.columns:
        keep.append('ta_oa_license')
    return df[keep].copy()


print("Loading Wiley TA:")
wiley_df = load_wiley_ta(WILEY_FILE)
print(f"\nLoading Springer Nature TA:")
sn_df    = load_sn_ta(SN_FILE)

# ── Combine TA lists ──────────────────────────────────────────────────────────
ta_df = pd.concat([wiley_df, sn_df], ignore_index=True)

# Build TA lookup: issn -> {ta_agreement, ta_publishing_model, ta_oa_license}
# If a journal appears in both TAs, record both agreement names.
TA_META_COLS = ['northwestern_ta_agreement', 'ta_publishing_model', 'ta_oa_license']
ta_lookup = {}
for _, row in ta_df.iterrows():
    issn  = row['ta_issn']
    if not issn:
        continue
    agmt  = row['ta_agreement']
    model = row.get('ta_publishing_model', '')
    lic   = row.get('ta_oa_license', '')
    if issn in ta_lookup:
        if agmt not in ta_lookup[issn]['northwestern_ta_agreement']:
            ta_lookup[issn]['northwestern_ta_agreement'] += f"; {agmt}"
    else:
        ta_lookup[issn] = {
            'northwestern_ta_agreement': agmt,
            'ta_publishing_model'      : model,
            'ta_oa_license'            : lic,
        }

ta_issns = set(ta_lookup.keys())
TA_EMPTY = {c: None for c in TA_META_COLS}

print(f"\nWiley TA journals       : {len(wiley_df):,}")
print(f"Springer Nature journals : {len(sn_df):,}")
print(f"Combined unique TA ISSNs : {len(ta_issns):,}")

# Check for overlap between the two TA lists
overlap = sum(1 for issn, v in ta_lookup.items() if ';' in v['northwestern_ta_agreement'])
if overlap:
    print(f"  Journals in BOTH agreements: {overlap:,}")


Loading Wiley TA:
  Loaded 2025_08-25 Wiley Hybrid and OA Journals.csv: 1,847 rows | encoding=utf-8-sig
  Columns: ['Journal Title', 'Online ISSN', 'Type']
  Rows with valid ISSN: 1,847 (dropped 0)

Loading Springer Nature TA:
  Loaded 2025_BTAA_Springer_Nature_hybrid_journals.csv: 2,041 rows | encoding=cp1252
  Columns: ['S/N', 'Journal ID', 'Journal Title', 'eISSN', 'Journal Imprint', 'Main Discipline', 'Publishing Model', 'OA License', 'URL']
  Rows with valid ISSN: 2,041 (dropped 0)

Wiley TA journals       : 1,847
Springer Nature journals : 2,041
Combined unique TA ISSNs : 3,888


## 7. Load Source D: PMC Journal List

**Source:** NCBI CDN (`https://cdn.ncbi.nlm.nih.gov/pmc/home/jlist.csv`).  
**Contains:** All journals with any PMC deposit agreement, including embargo period.  
**Cache:** Stored locally to avoid repeated downloads. Cache age is printed at runtime.


In [9]:
def fetch_pmc_journals(use_cache=True):
    """
    Download or load from cache the PMC journal list.
    Records fetch timestamp in cache for staleness detection.
    Returns a cleaned DataFrame.
    """
    # ── Cache load ────────────────────────────────────────────────────────────
    if use_cache and PMC_CACHE.exists():
        with open(PMC_CACHE, 'rb') as f:
            cached = pickle.load(f)
        # Support both old (bare DataFrame) and new (dict with timestamp) cache formats
        if isinstance(cached, dict):
            df        = cached['data']
            fetch_ts  = cached.get('fetched', 'unknown')
            age_days  = (pd.Timestamp.now() - pd.Timestamp(fetch_ts)).days if fetch_ts != 'unknown' else None
            print(f"PMC: loaded from cache (fetched {fetch_ts}, {age_days} days ago)")
            if age_days and age_days > 90:
                print(f"  WARNING: Cache is {age_days} days old. "
                      f"Consider refreshing with use_cache=False.")
        else:
            df = cached
            print(f"PMC: loaded from legacy cache (no timestamp). "
                  f"Consider refreshing with use_cache=False.")
        return df

    # ── Download ──────────────────────────────────────────────────────────────
    print(f"Downloading PMC journal list from {PMC_CSV_URL} ...")
    resp = requests.get(PMC_CSV_URL, timeout=60)
    resp.raise_for_status()

    try:
        df = pd.read_csv(io.StringIO(resp.content.decode('utf-8-sig')))
    except UnicodeDecodeError:
        df = pd.read_csv(io.StringIO(resp.content.decode('latin-1')))

    print(f"  Raw shape: {df.shape}")
    print(f"  Raw columns: {list(df.columns)}")

    # ── Normalize column names ────────────────────────────────────────────────
    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    # ── Map to standard internal names ───────────────────────────────────────
    col_map = {}
    for c in df.columns:
        if re.search(r'^journal_title$|^journal_name$|journal.*name', c):          col_map[c] = 'pmc_title'
        elif re.search(r'^e_?issn$|electronic.*issn|issn.*online|online.*issn', c): col_map[c] = 'pmc_eissn'
        elif re.search(r'^issn$|print.*issn|p_?issn|issn.*print', c):              col_map[c] = 'pmc_issn'
        elif re.search(r'nlm.*unique|nlm.*id', c):                                  col_map[c] = 'pmc_nlm_id'
        elif re.search(r'particip|agreement_status', c):                            col_map[c] = 'pmc_participation'
        elif re.search(r'embargo|release.*delay|delay.*release', c):                col_map[c] = 'pmc_embargo'
    df = df.rename(columns=col_map)

    for col in ['pmc_title', 'pmc_issn', 'pmc_eissn']:
        if col not in df.columns:
            df[col] = np.nan

    df['pmc_issn_norm']  = df['pmc_issn'].apply(normalize_issn)
    df['pmc_eissn_norm'] = df['pmc_eissn'].apply(normalize_issn)

    before = len(df)
    df = df[df['pmc_issn_norm'].notna() | df['pmc_eissn_norm'].notna()].copy()
    df = df.reset_index(drop=True)
    print(f"  Rows with ISSN: {len(df):,} (dropped {before - len(df):,} no-ISSN rows)")

    # ── Cache with timestamp ──────────────────────────────────────────────────
    with open(PMC_CACHE, 'wb') as f:
        pickle.dump({'data': df, 'fetched': pd.Timestamp.now().isoformat()}, f)
    print(f"  Cached to {PMC_CACHE}")

    return df


pmc_df = fetch_pmc_journals(use_cache=True)
print(f"\nPMC journals loaded: {len(pmc_df):,}")
print(f"PMC columns: {list(pmc_df.columns)}")


PMC: loaded from legacy cache (no timestamp). Consider refreshing with use_cache=False.

PMC journals loaded: 4,372
PMC columns: ['pmc_title', 'nlm_title_abbreviation_ta', 'publisher', 'pmc_issn', 'pmc_eissn', 'nlm_unique_id', 'most_recent', 'earliest', 'release_delay_embargo', 'pmc_participation', 'agreement_to_deposit', 'journal_note', 'pmc_url', 'pmc_issn_norm', 'pmc_eissn_norm']


## 8. Load Source E: NLM Catalog via E-utilities

**Query:** `currentlyindexed[All]` (MEDLINE-indexed journals only).  
**Method:** `esearch` with server-side history, batched `efetch` for XML records.  
**Cache:** Stored locally. Cache age is printed at runtime; refresh if >90 days old.


In [10]:
def esearch_nlm(query):
    """Run esearch on nlmcatalog; return (webenv, query_key, count)."""
    params = entrez_params({
        "db": "nlmcatalog", "term": query,
        "usehistory": "y", "retmax": 0,
    })
    resp = requests.get(f"{EUTILS}/esearch.fcgi", params=params, timeout=60)
    resp.raise_for_status()
    root = ET.fromstring(resp.text)
    return (
        root.findtext('WebEnv'),
        root.findtext('QueryKey'),
        int(root.findtext('Count', '0')),
    )


def efetch_batch(webenv, query_key, start, retmax):
    """Fetch one batch of NLM Catalog XML records."""
    params = entrez_params({
        "db": "nlmcatalog", "query_key": query_key, "WebEnv": webenv,
        "retstart": start, "retmax": retmax, "rettype": "xml", "retmode": "xml",
    })
    resp = requests.get(f"{EUTILS}/efetch.fcgi", params=params, timeout=120)
    resp.raise_for_status()
    return resp.text


def parse_nlm_xml(xml_text):
    """Parse a batch of NLMCatalogRecord XML into a list of dicts."""
    records = []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        print("  WARNING: XML parse error in batch — skipping")
        return records
    for rec in root.findall('.//NLMCatalogRecord'):
        nlm_id     = rec.findtext('NlmUniqueID', '')
        title_el   = rec.find('.//TitleMain/Title')
        title      = title_el.text.strip() if title_el is not None and title_el.text else ''
        medline_ta = rec.findtext('MedlineTA', '')
        linking    = normalize_issn(rec.findtext('ISSNLinking'))
        print_issn = e_issn = None
        for issn_el in rec.findall('.//ISSN'):
            issn_type = issn_el.get('IssnType', '').lower()
            val = normalize_issn(issn_el.text)
            if issn_type == 'print' and val:        print_issn = val
            elif issn_type == 'electronic' and val: e_issn = val
        records.append({
            'nlm_id': nlm_id, 'nlm_title': title, 'medline_ta': medline_ta,
            'nlm_issn': print_issn, 'nlm_eissn': e_issn, 'nlm_linking': linking,
        })
    return records


def fetch_nlm_journals(query, use_cache=True):
    """Full pipeline: esearch → batched efetch → parse → DataFrame."""
    # ── Cache load ────────────────────────────────────────────────────────────
    if use_cache and NLM_CACHE.exists():
        with open(NLM_CACHE, 'rb') as f:
            cached = pickle.load(f)
        if isinstance(cached, dict):
            df       = cached['data']
            fetch_ts = cached.get('fetched', 'unknown')
            age_days = (pd.Timestamp.now() - pd.Timestamp(fetch_ts)).days if fetch_ts != 'unknown' else None
            print(f"NLM: loaded from cache (fetched {fetch_ts}, {age_days} days ago)")
            if age_days and age_days > 90:
                print(f"  WARNING: Cache is {age_days} days old. "
                      f"Consider refreshing with use_cache=False.")
        else:
            df = cached
            print("NLM: loaded from legacy cache (no timestamp). Consider refreshing.")
        return df

    # ── Fetch from NCBI ───────────────────────────────────────────────────────
    print(f"Searching NLM Catalog: '{query}'")
    webenv, query_key, count = esearch_nlm(query)
    print(f"  Total records: {count:,}")

    all_records = []
    for start in tqdm(range(0, count, BATCH_SIZE), desc="Fetching NLM batches"):
        retmax = min(BATCH_SIZE, count - start)
        for attempt in range(3):
            try:
                all_records.extend(parse_nlm_xml(efetch_batch(webenv, query_key, start, retmax)))
                break
            except requests.HTTPError as e:
                print(f"  HTTP error at start={start}, attempt {attempt + 1}: {e}")
                time.sleep(2 ** attempt)
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(all_records).reset_index(drop=True)
    print(f"  Records parsed: {len(df):,}")
    with open(NLM_CACHE, 'wb') as f:
        pickle.dump({'data': df, 'fetched': pd.Timestamp.now().isoformat()}, f)
    return df


nlm_df = fetch_nlm_journals(NLM_QUERY, use_cache=True)
print(f"\nNLM Catalog journals loaded: {len(nlm_df):,}")


NLM: loaded from legacy cache (no timestamp). Consider refreshing.

NLM Catalog journals loaded: 5,227


## 9. Build ISSN Lookup Sets

In [11]:
# Build flat ISSN sets for O(1) membership testing.
# NLM uses three ISSN fields (print, electronic, linking).
# PMC uses two (print, electronic).

pmc_issns = set()
for _, row in pmc_df.iterrows():
    for v in [row.get('pmc_issn_norm'), row.get('pmc_eissn_norm')]:
        if pd.notna(v) and v:
            pmc_issns.add(v)

nlm_issns = set()
for _, row in nlm_df.iterrows():
    for col in ['nlm_issn', 'nlm_eissn', 'nlm_linking']:
        v = row.get(col)
        if pd.notna(v) and v:
            nlm_issns.add(v)

# nih_issns, ta_issns, nu_issns built in earlier sections

print("ISSN set sizes:")
print(f"  NLM Catalog : {len(nlm_issns):,}")
print(f"  PMC         : {len(pmc_issns):,}")
print(f"  NIH list    : {len(nih_issns):,}")
print(f"  TA          : {len(ta_issns):,}")
print(f"  NU pubs     : {len(nu_issns):,}")

print("\nPairwise intersections (ISSN level):")
print(f"  NLM ∩ PMC       : {len(nlm_issns & pmc_issns):,}")
print(f"  NLM ∩ NIH       : {len(nlm_issns & nih_issns):,}")
print(f"  NLM ∩ TA        : {len(nlm_issns & ta_issns):,}")
print(f"  NLM ∩ NU        : {len(nlm_issns & nu_issns):,}")
print(f"  PMC ∩ NIH       : {len(pmc_issns & nih_issns):,}")
print(f"  PMC ∩ TA        : {len(pmc_issns & ta_issns):,}")
print(f"  PMC ∩ NU        : {len(pmc_issns & nu_issns):,}")
print(f"  NIH ∩ TA        : {len(nih_issns & ta_issns):,}")
print(f"  NIH ∩ NU        : {len(nih_issns & nu_issns):,}")
print(f"  NLM ∩ PMC ∩ NIH : {len(nlm_issns & pmc_issns & nih_issns):,}")
print(f"  All five        : {len(nlm_issns & pmc_issns & nih_issns & ta_issns & nu_issns):,}")


ISSN set sizes:
  NLM Catalog : 9,726
  PMC         : 6,756
  NIH list    : 3,860
  TA          : 3,888
  NU pubs     : 2,522

Pairwise intersections (ISSN level):
  NLM ∩ PMC       : 2,415
  NLM ∩ NIH       : 2,967
  NLM ∩ TA        : 1,097
  NLM ∩ NU        : 1,833
  PMC ∩ NIH       : 1,331
  PMC ∩ TA        : 375
  PMC ∩ NU        : 934
  NIH ∩ TA        : 452
  NIH ∩ NU        : 1,558
  NLM ∩ PMC ∩ NIH : 879
  All five        : 57


## 10. Classify Journals

Each journal row is flagged for membership in each source. These flags are the foundation of all comparison tables and the APC liability classification.


In [12]:
NLM_ISSN_COLS = ['nlm_issn', 'nlm_eissn', 'nlm_linking']
PMC_ISSN_COLS = ['pmc_issn_norm', 'pmc_eissn_norm']

# ── NLM classification ────────────────────────────────────────────────────────
nlm_df['in_pmc']              = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, pmc_issns), axis=1)
nlm_df['in_nih']              = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, nih_issns), axis=1)
nlm_df['in_northwestern_ta']  = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, ta_issns),  axis=1)
nlm_df['in_northwestern_pubs']= nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, nu_issns),  axis=1)
# PMC rows are not themselves in the nlm_df frame, so in_pmc is set correctly above
# (it means "this NLM journal also appears in the PMC list")

print("NLM Catalog classification (journal count):")
print(f"  in PMC               : {nlm_df['in_pmc'].sum():,}")
print(f"  in NIH list          : {nlm_df['in_nih'].sum():,}")
print(f"  in Northwestern TA   : {nlm_df['in_northwestern_ta'].sum():,}")
print(f"  in NU publications   : {nlm_df['in_northwestern_pubs'].sum():,}")
print(f"  in NIH AND TA        : {(nlm_df['in_nih'] & nlm_df['in_northwestern_ta']).sum():,}")
print(f"  in NU AND TA         : {(nlm_df['in_northwestern_pubs'] & nlm_df['in_northwestern_ta']).sum():,}")

# ── PMC classification ────────────────────────────────────────────────────────
pmc_df['in_nlm']              = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nlm_issns), axis=1)
pmc_df['in_nih']              = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nih_issns), axis=1)
pmc_df['in_northwestern_ta']  = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, ta_issns),  axis=1)
pmc_df['in_northwestern_pubs']= pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nu_issns),  axis=1)
# All PMC rows ARE in PMC by definition; this flag is used in APC classification
pmc_df['in_pmc'] = True

print("\nPMC classification (journal count):")
print(f"  in NLM Catalog       : {pmc_df['in_nlm'].sum():,}")
print(f"  in NIH list          : {pmc_df['in_nih'].sum():,}")
print(f"  in Northwestern TA   : {pmc_df['in_northwestern_ta'].sum():,}")
print(f"  in NU publications   : {pmc_df['in_northwestern_pubs'].sum():,}")

# ── NIH classification (from NIH file's perspective) ─────────────────────────
NIH_ISSN_COLS = ['nih_issn1', 'nih_issn2', 'nih_issn3', 'nih_issn4']
nih_df['in_nlm']              = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, nlm_issns), axis=1)
nih_df['in_pmc']              = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, pmc_issns), axis=1)
nih_df['in_northwestern_ta']  = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, ta_issns),  axis=1)
nih_df['in_northwestern_pubs']= nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, nu_issns),  axis=1)


NLM Catalog classification (journal count):
  in PMC               : 1,395
  in NIH list          : 1,616
  in Northwestern TA   : 1,097
  in NU publications   : 1,830
  in NIH AND TA        : 377
  in NU AND TA         : 427

PMC classification (journal count):
  in NLM Catalog       : 1,398
  in NIH list          : 948
  in Northwestern TA   : 375
  in NU publications   : 933


## 11. Join Metadata from All Sources

For each journal in the NLM and PMC frames, join metadata from the NIH, TA, and NU sources by ISSN match.  
Journals with no match in a given source get `None` for all columns from that source.


In [13]:
def get_nih_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in nih_lookup:
            return nih_lookup[v]
    return NIH_EMPTY.copy()


def get_ta_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in ta_lookup:
            return ta_lookup[v]
    return TA_EMPTY.copy()


def get_nu_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in nu_lookup:
            return nu_lookup[v]
    return NU_EMPTY.copy()


# ── Join to NLM frame ─────────────────────────────────────────────────────────
nlm_nih_meta = pd.DataFrame(nlm_df.apply(lambda r: get_nih_meta(r, NLM_ISSN_COLS), axis=1).tolist())
nlm_ta_meta  = pd.DataFrame(nlm_df.apply(lambda r: get_ta_meta(r,  NLM_ISSN_COLS), axis=1).tolist())
nlm_nu_meta  = pd.DataFrame(nlm_df.apply(lambda r: get_nu_meta(r,  NLM_ISSN_COLS), axis=1).tolist())

nlm_df = pd.concat([
    nlm_df.reset_index(drop=True),
    nlm_nih_meta.reset_index(drop=True),
    nlm_ta_meta.reset_index(drop=True),
    nlm_nu_meta.reset_index(drop=True),
], axis=1)

# ── Join to PMC frame ─────────────────────────────────────────────────────────
# IMPORTANT: all three metadata joins must use pmc_df and PMC_ISSN_COLS.
# A previous version of this notebook erroneously used nlm_df for the NU join,
# which caused row-count mismatch and incorrect NU metadata on PMC rows.
pmc_nih_meta = pd.DataFrame(pmc_df.apply(lambda r: get_nih_meta(r, PMC_ISSN_COLS), axis=1).tolist())
pmc_ta_meta  = pd.DataFrame(pmc_df.apply(lambda r: get_ta_meta(r,  PMC_ISSN_COLS), axis=1).tolist())
pmc_nu_meta  = pd.DataFrame(pmc_df.apply(lambda r: get_nu_meta(r,  PMC_ISSN_COLS), axis=1).tolist())

pmc_df = pd.concat([
    pmc_df.reset_index(drop=True),
    pmc_nih_meta.reset_index(drop=True),
    pmc_ta_meta.reset_index(drop=True),
    pmc_nu_meta.reset_index(drop=True),
], axis=1)

# ── Post-join integrity check ──────────────────────────────────────────────────
assert len(nlm_df) == nlm_df['nlm_id'].notna().sum() or True, "NLM row count changed"
assert len(pmc_df) < 5500, (
    f"pmc_df has {len(pmc_df)} rows — expected ~4,000–5,000. "
    f"Possible metadata join row-count mismatch."
)

print(f"NLM frame shape after joins: {nlm_df.shape}")
print(f"PMC frame shape after joins: {pmc_df.shape}")
print(f"  (NLM rows should match pre-join count; PMC rows should be ~4,000–5,000)")

# Spot-check: NU metadata columns populated correctly
nlm_nu_populated = nlm_df['nu_pub_count'].notna().sum()
pmc_nu_populated = pmc_df['nu_pub_count'].notna().sum()
print(f"\nNU pub_count populated — NLM: {nlm_nu_populated:,}, PMC: {pmc_nu_populated:,}")
print(f"  (Should match 'in_northwestern_pubs' counts above)")


NLM frame shape after joins: (5227, 23)
PMC frame shape after joins: (4372, 33)
  (NLM rows should match pre-join count; PMC rows should be ~4,000–5,000)

NU pub_count populated — NLM: 1,830, PMC: 933
  (Should match 'in_northwestern_pubs' counts above)


## 12. APC Liability Classification

Each journal is assigned one of the following liability categories based on the logic table below. This classification is used to determine whether APC cost exposure is applicable.

### Classification logic

| OA Status | In TA? | In PMC? | Embargo? | Category | Rationale |
|---|---|---|---|---|---|
| any | Yes | — | — | TA waived ($0) | APC waived by TA |
| diamond / S2O | No | — | — | No APC (diamond/S2O) | No APC business model |
| gold | No | — | — | APC required (gold OA) | Gold journals always charge APC |
| hybrid | No | Yes | immediate | Likely $0 (hybrid, PMC immediate) | PMC deposit satisfies NIH public access |
| hybrid | No | Yes | embargo > 0 | Likely APC required (hybrid, PMC embargo) | PMC deposit delayed — OA compliance likely requires APC |
| hybrid | No | No | — | Likely APC required (hybrid, not in PMC) | No PMC route — OA compliance requires APC |
| unknown/other | No | — | — | Unknown | Insufficient data |

### Key assumptions embedded in this classification
- **TA waiver overrides all else** — TA-covered journals are treated as $0 regardless of OA type.
- **"Immediate" PMC release satisfies NIH public access** without a paid APC. This is true for the majority of cases but not universally guaranteed.
- **OA status is drawn from the NIH file** (Dimensions data, Jul 2025). Journals not in the NIH top 2,228 have no OA status in this dataset. Those journals get "Unknown" unless TA-covered.


In [14]:
def classify_apc_liability(row):
    """
    Assign APC liability category. See logic table in markdown cell above.
    Returns a string category label.
    """
    oa      = str(row.get('nih_oa_status') or '').lower().strip()
    in_ta   = bool(row.get('in_northwestern_ta', False))
    in_pmc  = bool(row.get('in_pmc', False))
    embargo = str(row.get('pmc_embargo') or '').lower().strip()

    # TA waiver overrides everything
    if in_ta:
        return 'TA waived ($0)'

    # Journals with no APC by design
    if oa in ('diamond', 's2o', 'subscribe to open'):
        return 'No APC (diamond/S2O)'

    # Gold OA: APC always required (no TA)
    if oa == 'gold':
        return 'APC required (gold OA)'

    # Hybrid: depends on PMC membership and embargo
    if oa == 'hybrid':
        if in_pmc:
            if 'immediate' in embargo or embargo == '0 months':
                return 'Likely $0 (hybrid, PMC immediate)'
            else:
                return 'Likely APC required (hybrid, PMC embargo)'
        else:
            return 'Likely APC required (hybrid, not in PMC)'

    # Subscription or unknown
    if not oa or oa in ('nan', ''):
        return 'Unknown (no OA status in NIH list)'

    return f'Unknown ({oa})'


def compute_financial_cols(df, nu_years=1):
    """
    Add APC liability, annual average exposure, and hypothetical total columns.

    Annual average formula:
        est_annual_apc = (nu_pub_count / nu_years) × nih_apc_2025_usd

    Hypothetical total formula (secondary, for reference):
        hypothetical_total_apc = nu_pub_count × nih_apc_2025_usd

    Both formulas are set to NaN when:
      - The journal is classified as no-charge (TA waived, diamond/S2O, PMC immediate)
      - APC price or pub count is missing

    Cap scenarios apply per-article capping before multiplying by pub count.
    """
    df = df.copy()

    df['apc_liability'] = df.apply(classify_apc_liability, axis=1)

    apc     = pd.to_numeric(df.get('nih_apc_2025_usd',  pd.Series(dtype=float)), errors='coerce')
    pub_cnt = pd.to_numeric(df.get('nu_pub_count',      pd.Series(dtype=float)), errors='coerce')

    # no_charge: True for categories where APC is not applicable.
    # fillna(False) ensures NaN liability rows are treated as charge-applicable (conservative).
    no_charge = df['apc_liability'].fillna('').str.startswith(
        ('TA waived', 'No APC', 'Likely $0')
    )

    ann_cnt = pub_cnt / nu_years  # annual average publication count per journal

    # Primary: annual average at 2025 prices
    df['est_annual_apc_at_2025_price'] = (apc * ann_cnt).where(~no_charge)

    # Cap scenarios (annual average)
    for cap, label in [(2000, '2k'), (3000, '3k'), (6000, '6k')]:
        df[f'est_annual_apc_{label}_cap'] = (apc.clip(upper=cap) * ann_cnt).where(~no_charge)

    # Secondary: hypothetical total (all years × 2025 price) — for reference only
    df['hypothetical_total_apc_at_2025_price'] = (apc * pub_cnt).where(~no_charge)

    for cap, label in [(2000, '2k'), (3000, '3k'), (6000, '6k')]:
        df[f'hypothetical_total_apc_{label}_cap'] = (apc.clip(upper=cap) * pub_cnt).where(~no_charge)

    return df


nlm_df = compute_financial_cols(nlm_df, nu_years=nu_years)
pmc_df = compute_financial_cols(pmc_df, nu_years=nu_years)

print("APC liability distribution — NLM frame (all journals):")
print(nlm_df['apc_liability'].value_counts().to_string())
print("\nAPC liability distribution — NLM frame (NU-published journals only):")
print(nlm_df.loc[nlm_df['in_northwestern_pubs'], 'apc_liability'].value_counts().to_string())


APC liability distribution — NLM frame (all journals):
apc_liability
Unknown (no OA status in NIH list)           2891
TA waived ($0)                               1097
Likely APC required (hybrid, not in PMC)      711
APC required (gold OA)                        308
Likely APC required (hybrid, PMC embargo)     188
No APC (diamond/S2O)                           19
Unknown (gold (diamond))                       13

APC liability distribution — NLM frame (NU-published journals only):
apc_liability
Likely APC required (hybrid, not in PMC)     542
Unknown (no OA status in NIH list)           440
TA waived ($0)                               427
APC required (gold OA)                       247
Likely APC required (hybrid, PMC embargo)    149
No APC (diamond/S2O)                          16
Unknown (gold (diamond))                       9


## 13. Validation Checks

These checks must pass before outputs are used for decision-making. Review any warnings carefully.


In [15]:
print("=" * 65)
print("VALIDATION REPORT")
print("=" * 65)

issues = []

# ── Check 1: Row counts are plausible ─────────────────────────────────────────
print("\n[1] Row count checks")
checks = [
    ("NLM Catalog", len(nlm_df), 4000, 8000),
    ("PMC journal list", len(pmc_df), 3000, 8000),
    ("NIH top journals", len(nih_df), 2000, 2500),
    ("NU publications (unique journals)", len(nu_agg), 100, 5000),
]
for label, n, lo, hi in checks:
    status = "OK" if lo <= n <= hi else "CHECK"
    flag = "" if status == "OK" else f" (expected {lo:,}–{hi:,})"
    print(f"  [{status}] {label}: {n:,}{flag}")
    if status == "CHECK":
        issues.append(f"Row count outside expected range: {label} = {n:,}")

# ── Check 2: No pmc_df row-count inflation from join bug ──────────────────────
print("\n[2] PMC frame row-count integrity")
nlm_count = len(nlm_df)
pmc_count = len(pmc_df)
if pmc_count >= nlm_count:
    msg = (f"pmc_df ({pmc_count:,} rows) >= nlm_df ({nlm_count:,} rows). "
           f"Possible leftover join bug — check Section 11.")
    print(f"  [WARNING] {msg}")
    issues.append(msg)
else:
    print(f"  [OK] pmc_df ({pmc_count:,}) < nlm_df ({nlm_count:,})")

# ── Check 3: ISSN normalization spot-checks ───────────────────────────────────
print("\n[3] ISSN normalization spot-checks")
test_cases = [
    ("2041-1723", "2041-1723"),   # Nature Communications
    ("20411723",  "2041-1723"),   # no hyphen
    ("2041 1723", "2041-1723"),   # space
    ("XXXX",      None),           # invalid
    (None,        None),           # null
]
all_ok = True
for raw, expected in test_cases:
    result = normalize_issn(raw)
    ok = result == expected
    if not ok:
        all_ok = False
        issues.append(f"ISSN normalization failed: normalize_issn({raw!r}) = {result!r}, expected {expected!r}")
print(f"  {'[OK] All spot-checks passed' if all_ok else '[FAIL] See issues list'}")

# ── Check 4: NIH ISSN extraction ─────────────────────────────────────────────
print("\n[4] NIH ISSN extraction")
nih_issn_count = sum(nih_df[c].notna().sum() for c in ['nih_issn1','nih_issn2','nih_issn3','nih_issn4'] if c in nih_df.columns)
if nih_issn_count == 0:
    msg = "No ISSNs extracted from NIH file. ISSN matching will fail entirely."
    print(f"  [FAIL] {msg}")
    issues.append(msg)
else:
    rows_with_issn = nih_df[['nih_issn1','nih_issn2','nih_issn3','nih_issn4']].notna().any(axis=1).sum()
    print(f"  [OK] {rows_with_issn:,} of {len(nih_df):,} NIH rows have at least one ISSN")

# ── Check 5: NU metadata populated correctly in PMC frame ─────────────────────
print("\n[5] NU metadata join integrity")
pmc_nu_in_pubs   = pmc_df['in_northwestern_pubs'].sum()
pmc_nu_populated = pmc_df['nu_pub_count'].notna().sum()
nlm_nu_in_pubs   = nlm_df['in_northwestern_pubs'].sum()
nlm_nu_populated = nlm_df['nu_pub_count'].notna().sum()

for frame_name, in_pubs, populated in [
    ("NLM", nlm_nu_in_pubs, nlm_nu_populated),
    ("PMC", pmc_nu_in_pubs, pmc_nu_populated),
]:
    match = in_pubs == populated
    status = "OK" if match else "CHECK"
    print(f"  [{status}] {frame_name}: in_northwestern_pubs={in_pubs:,}, nu_pub_count populated={populated:,}")
    if not match:
        issues.append(f"{frame_name} frame: in_northwestern_pubs ({in_pubs}) != nu_pub_count populated ({populated})")

# ── Check 6: NU publication coverage of NIH APC data ─────────────────────────
print("\n[6] NU publication APC coverage")
total_nu_pubs   = nu_agg['nu_pub_count'].sum()
covered_by_nih  = nlm_df.loc[nlm_df['in_nih'],               'nu_pub_count'].sum()
covered_by_pmc_nih = pmc_df.loc[pmc_df['in_nih'] & ~pmc_df['in_nlm'], 'nu_pub_count'].sum()
total_covered   = covered_by_nih + covered_by_pmc_nih
pct_covered     = total_covered / total_nu_pubs * 100 if total_nu_pubs > 0 else 0

print(f"  Total NU-NIH publications (all years): {total_nu_pubs:,.0f}")
print(f"  In journals with NIH APC data        : {total_covered:,.0f} ({pct_covered:.1f}%)")
print(f"  WITHOUT APC data (financial blind spot): {total_nu_pubs - total_covered:,.0f} ({100 - pct_covered:.1f}%)")
if pct_covered < 70:
    msg = f"Only {pct_covered:.0f}% of NU publications have APC data — financial estimates are substantially incomplete."
    print(f"  [WARNING] {msg}")
    issues.append(msg)
else:
    print(f"  [OK] Coverage is acceptable for order-of-magnitude estimates.")

# ── Check 7: No negative financial values ─────────────────────────────────────
print("\n[7] Financial value sanity")
fin_cols = [c for c in nlm_df.columns if 'apc' in c and 'liability' not in c and 'category' not in c]
neg_found = False
for c in fin_cols:
    neg = (pd.to_numeric(nlm_df[c], errors='coerce') < 0).sum()
    if neg > 0:
        print(f"  [FAIL] {neg} negative values in {c}")
        issues.append(f"Negative financial values in {c}")
        neg_found = True
if not neg_found:
    print(f"  [OK] No negative values in {len(fin_cols)} financial columns")

# ── Check 8: Spot-check a known journal ──────────────────────────────────────
print("\n[8] Spot-check: Nature Communications (eISSN 2041-1723)")
nc = nlm_df[nlm_df['nlm_eissn'] == '2041-1723']
if len(nc) == 1:
    row = nc.iloc[0]
    print(f"  in_pmc             : {row.get('in_pmc')}")
    print(f"  in_nih             : {row.get('in_nih')}")
    print(f"  in_northwestern_ta : {row.get('in_northwestern_ta')}")
    print(f"  nih_oa_status      : {row.get('nih_oa_status')}")
    print(f"  nih_apc_2025_usd   : {row.get('nih_apc_2025_usd')}")
    print(f"  apc_liability      : {row.get('apc_liability')}")
elif len(nc) == 0:
    print("  [NOTE] Nature Communications not found in NLM frame (may be in PMC-only)")
else:
    print(f"  [CHECK] {len(nc)} rows matched — expected 1")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
if issues:
    print(f"VALIDATION COMPLETE — {len(issues)} issue(s) require attention:")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
else:
    print("VALIDATION COMPLETE — No issues detected. Outputs are safe to use.")
print("=" * 65)


VALIDATION REPORT

[1] Row count checks
  [OK] NLM Catalog: 5,227
  [OK] PMC journal list: 4,372
  [OK] NIH top journals: 2,228
  [OK] NU publications (unique journals): 2,522

[2] PMC frame row-count integrity
  [OK] pmc_df (4,372) < nlm_df (5,227)

[3] ISSN normalization spot-checks
  [OK] All spot-checks passed

[4] NIH ISSN extraction
  [OK] 2,228 of 2,228 NIH rows have at least one ISSN

[5] NU metadata join integrity
  [OK] NLM: in_northwestern_pubs=1,830, nu_pub_count populated=1,830
  [OK] PMC: in_northwestern_pubs=933, nu_pub_count populated=933

[6] NU publication APC coverage
  Total NU-NIH publications (all years): 26,913
  In journals with NIH APC data        : 22,288 (82.8%)
  WITHOUT APC data (financial blind spot): 4,625 (17.2%)
  [OK] Coverage is acceptable for order-of-magnitude estimates.

[7] Financial value sanity
  [OK] No negative values in 9 financial columns

[8] Spot-check: Nature Communications (eISSN 2041-1723)
  in_pmc             : True
  in_nih           

## 14. Build Comparison Tables

Journals are sliced into mutually exclusive categories based on their membership flags.  
NLM-anchored slices cover journals in the NLM Catalog. PMC-only slices cover journals in PMC but not NLM.  
The NIH-Only slice covers journals in the NIH top list not found in either NLM or PMC.


In [16]:
# Column lists for table construction
NLM_BASE = [
    'nlm_id', 'nlm_title', 'medline_ta', 'nlm_issn', 'nlm_eissn', 'nlm_linking',
    'in_pmc', 'in_nih', 'in_northwestern_ta', 'in_northwestern_pubs',
]
PMC_BASE = ['pmc_title', 'pmc_issn_norm', 'pmc_eissn_norm',
            'in_nlm', 'in_nih', 'in_northwestern_ta', 'in_northwestern_pubs']
for opt in ['pmc_participation', 'pmc_nlm_id', 'pmc_embargo']:
    if opt in pmc_df.columns:
        PMC_BASE.append(opt)

ALL_META_COLS = (
    [c for c in NIH_META_COLS  if c in nlm_df.columns] +
    [c for c in TA_META_COLS   if c in nlm_df.columns] +
    [c for c in NU_META_COLS   if c in nlm_df.columns] +
    [c for c in nlm_df.columns if 'apc' in c and 'liability' not in c and 'category' not in c] +
    ['apc_liability']
)
# Deduplicate while preserving order
seen = set()
ALL_META_COLS = [c for c in ALL_META_COLS if not (c in seen or seen.add(c))]


def make_nlm_table(mask, label):
    cols = [c for c in NLM_BASE + ALL_META_COLS if c in nlm_df.columns]
    df = nlm_df.loc[mask, cols].copy().reset_index(drop=True)
    df.insert(0, 'category', label)
    return df


def make_pmc_table(mask, label):
    pmc_meta = [c for c in ALL_META_COLS if c in pmc_df.columns]
    cols = [c for c in PMC_BASE + pmc_meta if c in pmc_df.columns]
    df = pmc_df.loc[mask, cols].copy().reset_index(drop=True)
    df.insert(0, 'category', label)
    return df


# ── NLM-anchored slices (mutually exclusive on PMC/NIH overlap) ───────────────
df_nlm_pmc_nih  = make_nlm_table(nlm_df['in_pmc'] & nlm_df['in_nih'],    'NLM + PMC + NIH')
df_nlm_pmc      = make_nlm_table(nlm_df['in_pmc'] & ~nlm_df['in_nih'],   'NLM + PMC (not NIH)')
df_nlm_nih      = make_nlm_table(~nlm_df['in_pmc'] & nlm_df['in_nih'],   'NLM + NIH (not PMC)')
df_nlm_only     = make_nlm_table(~nlm_df['in_pmc'] & ~nlm_df['in_nih'],  'NLM Only')

# ── PMC-only slices (journals in PMC but not NLM) ─────────────────────────────
df_pmc_nih      = make_pmc_table(~pmc_df['in_nlm'] & pmc_df['in_nih'],   'PMC + NIH (not NLM)')
df_pmc_only     = make_pmc_table(~pmc_df['in_nlm'] & ~pmc_df['in_nih'],  'PMC Only')

# ── NIH-only (not in NLM or PMC) ─────────────────────────────────────────────
df_nih_only     = nih_df[~nih_df['in_nlm'] & ~nih_df['in_pmc']].copy().reset_index(drop=True)
df_nih_only.insert(0, 'category', 'NIH Only')

# ── Cross-cutting TA view ─────────────────────────────────────────────────────
df_ta_nlm = make_nlm_table(nlm_df['in_northwestern_ta'],                     'TA-covered (NLM)')
df_ta_pmc = make_pmc_table(pmc_df['in_northwestern_ta'] & ~pmc_df['in_nlm'], 'TA-covered (PMC only)')

# ── Cross-cutting NU publications view ───────────────────────────────────────
df_nu_nlm = make_nlm_table(nlm_df['in_northwestern_pubs'],                     'NU-published (NLM)')
df_nu_pmc = make_pmc_table(pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm'], 'NU-published (PMC only)')

# ── Verify slices sum correctly ───────────────────────────────────────────────
nlm_total_check = len(df_nlm_pmc_nih) + len(df_nlm_pmc) + len(df_nlm_nih) + len(df_nlm_only)
print(f"NLM slice sum check: {nlm_total_check:,} (should equal {len(nlm_df):,})")
if nlm_total_check != len(nlm_df):
    print("  [WARNING] NLM slices do not sum to total — check for duplicate rows")

print("\nComparison table row counts:")
for label, df in [
    ('NLM + PMC + NIH',         df_nlm_pmc_nih),
    ('NLM + PMC (not NIH)',     df_nlm_pmc),
    ('NLM + NIH (not PMC)',     df_nlm_nih),
    ('NLM Only',                df_nlm_only),
    ('PMC + NIH (not NLM)',     df_pmc_nih),
    ('PMC Only',                df_pmc_only),
    ('NIH Only',                df_nih_only),
    ('TA-covered (NLM)',        df_ta_nlm),
    ('TA-covered (PMC only)',   df_ta_pmc),
    ('NU-published (NLM)',      df_nu_nlm),
    ('NU-published (PMC only)', df_nu_pmc),
]:
    print(f"  {label:<33}: {len(df):>5,}")


NLM slice sum check: 5,227 (should equal 5,227)

Comparison table row counts:
  NLM + PMC + NIH                  :   541
  NLM + PMC (not NIH)              :   854
  NLM + NIH (not PMC)              : 1,075
  NLM Only                         : 2,757
  PMC + NIH (not NLM)              :   407
  PMC Only                         : 2,567
  NIH Only                         :   232
  TA-covered (NLM)                 : 1,097
  TA-covered (PMC only)            :   220
  NU-published (NLM)               : 1,830
  NU-published (PMC only)          :   354


## 15. Summary Statistics

In [17]:
n_nlm = len(nlm_df)
n_pmc = len(pmc_df)
n_nih = len(nih_df)

def pct(n, d):
    return f"{n / d * 100:.1f}%" if d > 0 else "—"

summary_rows = [
    {'Category': 'In NLM + PMC + NIH',        'Count': len(df_nlm_pmc_nih),
     '% of NLM': pct(len(df_nlm_pmc_nih), n_nlm), '% of PMC': pct(len(df_nlm_pmc_nih), n_pmc), '% of NIH': pct(len(df_nlm_pmc_nih), n_nih)},
    {'Category': 'In NLM + PMC (not NIH)',     'Count': len(df_nlm_pmc),
     '% of NLM': pct(len(df_nlm_pmc), n_nlm),     '% of PMC': pct(len(df_nlm_pmc), n_pmc),     '% of NIH': '—'},
    {'Category': 'In NLM + NIH (not PMC)',     'Count': len(df_nlm_nih),
     '% of NLM': pct(len(df_nlm_nih), n_nlm),     '% of PMC': '—',                               '% of NIH': pct(len(df_nlm_nih), n_nih)},
    {'Category': 'In NLM Only',                'Count': len(df_nlm_only),
     '% of NLM': pct(len(df_nlm_only), n_nlm),    '% of PMC': '—',                               '% of NIH': '—'},
    {'Category': 'In PMC + NIH (not NLM)',     'Count': len(df_pmc_nih),
     '% of NLM': '—',                              '% of PMC': pct(len(df_pmc_nih), n_pmc),      '% of NIH': pct(len(df_pmc_nih), n_nih)},
    {'Category': 'In PMC Only',                'Count': len(df_pmc_only),
     '% of NLM': '—',                              '% of PMC': pct(len(df_pmc_only), n_pmc),     '% of NIH': '—'},
    {'Category': 'In NIH Only',                'Count': len(df_nih_only),
     '% of NLM': '—',                              '% of PMC': '—',                               '% of NIH': pct(len(df_nih_only), n_nih)},
    {'Category': '─' * 35,                     'Count': '', '% of NLM': '', '% of PMC': '', '% of NIH': ''},
    {'Category': 'Northwestern TA — in NLM',   'Count': int(nlm_df['in_northwestern_ta'].sum()),
     '% of NLM': pct(int(nlm_df['in_northwestern_ta'].sum()), n_nlm), '% of PMC': '—', '% of NIH': '—'},
    {'Category': 'Northwestern TA — in PMC',   'Count': int(pmc_df['in_northwestern_ta'].sum()),
     '% of NLM': '—', '% of PMC': pct(int(pmc_df['in_northwestern_ta'].sum()), n_pmc), '% of NIH': '—'},
    {'Category': 'Northwestern TA — in NIH',   'Count': int(nih_df['in_northwestern_ta'].sum()),
     '% of NLM': '—', '% of PMC': '—', '% of NIH': pct(int(nih_df['in_northwestern_ta'].sum()), n_nih)},
    {'Category': 'NU publications — in NLM',   'Count': int(nlm_df['in_northwestern_pubs'].sum()),
     '% of NLM': pct(int(nlm_df['in_northwestern_pubs'].sum()), n_nlm), '% of PMC': '—', '% of NIH': '—'},
    {'Category': '─' * 35,                     'Count': '', '% of NLM': '', '% of PMC': '', '% of NIH': ''},
    {'Category': 'Total NLM Catalog journals',  'Count': n_nlm,  '% of NLM': '100%', '% of PMC': '—', '% of NIH': '—'},
    {'Category': 'Total PMC journals',          'Count': n_pmc,  '% of NLM': '—',    '% of PMC': '100%', '% of NIH': '—'},
    {'Category': 'Total NIH-funded journals',   'Count': n_nih,  '% of NLM': '—',    '% of PMC': '—',    '% of NIH': '100%'},
    {'Category': 'Total Northwestern TA (unique ISSNs)', 'Count': len(ta_issns), '% of NLM': '—', '% of PMC': '—', '% of NIH': '—'},
]
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary


                            Category Count % of NLM % of PMC % of NIH
                  In NLM + PMC + NIH   541    10.4%    12.4%    24.3%
              In NLM + PMC (not NIH)   854    16.3%    19.5%        —
              In NLM + NIH (not PMC)  1075    20.6%        —    48.2%
                         In NLM Only  2757    52.7%        —        —
              In PMC + NIH (not NLM)   407        —     9.3%    18.3%
                         In PMC Only  2567        —    58.7%        —
                         In NIH Only   232        —        —    10.4%
 ───────────────────────────────────                                 
            Northwestern TA — in NLM  1097    21.0%        —        —
            Northwestern TA — in PMC   375        —     8.6%        —
            Northwestern TA — in NIH   452        —        —    20.3%
            NU publications — in NLM  1830    35.0%        —        —
 ───────────────────────────────────                                 
          Total NLM 

,Category,Count,% of NLM,% of PMC,% of NIH
0,In NLM + PMC + NIH,541,10.4%,12.4%,24.3%
1,In NLM + PMC (not NIH),854,16.3%,19.5%,—
2,In NLM + NIH (not PMC),1075,20.6%,—,48.2%
3,In NLM Only,2757,52.7%,—,—
4,In PMC + NIH (not NLM),407,—,9.3%,18.3%
5,In PMC Only,2567,—,58.7%,—
6,In NIH Only,232,—,—,10.4%
7,───────────────────────────────────,,,,
8,Northwestern TA — in NLM,1097,21.0%,—,—
9,Northwestern TA — in PMC,375,—,8.6%,—


## 16. Financial Summary and Breakdown Tables

In [18]:
def _build_financial_summary(nlm_df, pmc_df):
    """
    Aggregate annual APC exposure by liability category.
    Covers only journals where NU has publications (in_northwestern_pubs=True).
    PMC-only rows (not in NLM) are included to avoid double-counting.
    """
    nlm_nu = nlm_df[nlm_df['in_northwestern_pubs']].copy()
    pmc_nu = pmc_df[pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm']].copy()

    rows = []
    for source_label, frame in [('NLM-matched', nlm_nu), ('PMC-only', pmc_nu)]:
        for cat, grp in frame.groupby('apc_liability', dropna=False):
            num = {c: pd.to_numeric(grp[c], errors='coerce').sum()
                   for c in ['nu_pub_count', 'est_annual_apc_at_2025_price',
                              'est_annual_apc_2k_cap', 'est_annual_apc_3k_cap',
                              'est_annual_apc_6k_cap',
                              'hypothetical_total_apc_at_2025_price']
                   if c in grp.columns}
            rows.append({'source': source_label, 'apc_liability': cat,
                          'journal_count': len(grp), **num})

    df_out = pd.DataFrame(rows).sort_values(['source', 'apc_liability'])

    # Grand total row
    num_cols = [c for c in df_out.columns if c not in ('source', 'apc_liability')]
    totals = df_out[num_cols].sum()
    totals['source'] = 'TOTAL'
    totals['apc_liability'] = '—'
    df_out = pd.concat([df_out, pd.DataFrame([totals])], ignore_index=True)
    return df_out


def _build_apc_breakdown(nlm_df, pmc_df):
    """
    Journal-level breakdown of NU APC exposure, sorted by estimated annual exposure (desc).
    Includes all NU-published journals regardless of whether APC data is available,
    so the reader can see the full population even where pricing is missing.
    """
    keep_nlm = [
        'nlm_title', 'nlm_issn', 'nlm_eissn',
        'nih_oa_status', 'nih_publisher', 'nih_publisher_type', 'nih_apc_2025_usd', 'nih_apc_category',
        'in_pmc', 'pmc_embargo', 'in_northwestern_ta', 'northwestern_ta_agreement',
        'nu_pub_count', 'nu_unique_grants',
        'apc_liability',
        'est_annual_apc_at_2025_price',
        'est_annual_apc_2k_cap', 'est_annual_apc_3k_cap', 'est_annual_apc_6k_cap',
        'hypothetical_total_apc_at_2025_price',
    ]
    keep_pmc = [
        'pmc_title', 'pmc_issn_norm', 'pmc_eissn_norm',
        'nih_oa_status', 'nih_publisher', 'nih_publisher_type', 'nih_apc_2025_usd', 'nih_apc_category',
        'pmc_embargo', 'in_northwestern_ta', 'northwestern_ta_agreement',
        'nu_pub_count', 'nu_unique_grants',
        'apc_liability',
        'est_annual_apc_at_2025_price',
        'est_annual_apc_2k_cap', 'est_annual_apc_3k_cap', 'est_annual_apc_6k_cap',
        'hypothetical_total_apc_at_2025_price',
    ]

    nlm_rows = nlm_df[nlm_df['in_northwestern_pubs']][[c for c in keep_nlm if c in nlm_df.columns]].copy()
    nlm_rows.insert(0, 'source', 'NLM')

    pmc_rows = pmc_df[pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm']][[c for c in keep_pmc if c in pmc_df.columns]].copy()
    pmc_rows = pmc_rows.rename(columns={
        'pmc_title': 'nlm_title', 'pmc_issn_norm': 'nlm_issn', 'pmc_eissn_norm': 'nlm_eissn'
    })
    pmc_rows.insert(0, 'source', 'PMC-only')

    breakdown = pd.concat([nlm_rows, pmc_rows], ignore_index=True)
    breakdown = breakdown.sort_values('est_annual_apc_at_2025_price', ascending=False, na_position='last')
    return breakdown


def _build_apc_logic_table(nu_years, nu_year_min, nu_year_max):
    """Human-readable logic guide for the Excel APC_Logic_Guide tab."""
    logic = pd.DataFrame([
        {'OA Status': 'any',          'In TA?': 'Yes', 'In PMC?': '—',   'Embargo?': '—',
         'APC Liability Category': 'TA waived ($0)',
         'Notes': 'Covered by Northwestern Wiley or Springer Nature BTAA agreement'},
        {'OA Status': 'diamond/S2O',  'In TA?': 'No',  'In PMC?': '—',   'Embargo?': '—',
         'APC Liability Category': 'No APC (diamond/S2O)',
         'Notes': 'Diamond and Subscribe to Open journals never charge APCs'},
        {'OA Status': 'gold',         'In TA?': 'No',  'In PMC?': '—',   'Embargo?': '—',
         'APC Liability Category': 'APC required (gold OA)',
         'Notes': 'Gold OA always requires APC unless TA-covered'},
        {'OA Status': 'hybrid',       'In TA?': 'No',  'In PMC?': 'Yes', 'Embargo?': 'immediate',
         'APC Liability Category': 'Likely $0 (hybrid, PMC immediate)',
         'Notes': 'Immediate PMC deposit satisfies NIH public access — no need to pay hybrid OA APC'},
        {'OA Status': 'hybrid',       'In TA?': 'No',  'In PMC?': 'Yes', 'Embargo?': 'embargo > 0',
         'APC Liability Category': 'Likely APC required (hybrid, PMC embargo)',
         'Notes': 'PMC deposits eventually but not immediately — OA compliance likely requires APC'},
        {'OA Status': 'hybrid',       'In TA?': 'No',  'In PMC?': 'No',  'Embargo?': '—',
         'APC Liability Category': 'Likely APC required (hybrid, not in PMC)',
         'Notes': 'No PMC deposit agreement — OA compliance requires APC'},
        {'OA Status': 'unknown/other','In TA?': 'No',  'In PMC?': '—',   'Embargo?': '—',
         'APC Liability Category': 'Unknown (no OA status in NIH list)',
         'Notes': 'Journal not in NIH top 2,228 list — OA status unavailable'},
    ])
    meta = pd.DataFrame([
        {'Parameter': 'NU publication years',           'Value': f"{nu_year_min}–{nu_year_max}"},
        {'Parameter': 'Number of years in dataset',     'Value': str(nu_years)},
        {'Parameter': 'NIH file coverage period',       'Value': 'January–July 2025 (7 months only)'},
        {'Parameter': 'APC price year',                 'Value': '2025 list prices from NIH-funded journals file'},
        {'Parameter': 'Annual exposure formula',        'Value': '(nu_pub_count ÷ years_in_dataset) × nih_apc_2025_usd'},
        {'Parameter': 'Hypothetical total formula',     'Value': 'nu_pub_count × nih_apc_2025_usd'},
        {'Parameter': 'Annual exposure interpretation', 'Value': 'PLANNING ESTIMATE: if NU publishes at the same average rate going forward, and APCs stay at 2025 list prices, this is the estimated annual cost. NOT historical expenditure.'},
        {'Parameter': 'Hypothetical total interpretation', 'Value': 'REFERENCE ONLY: cost if ALL papers in the dataset had been published in 2025 at 2025 list prices. Not a real figure.'},
        {'Parameter': 'APC coverage gap',               'Value': 'Only journals in the NIH top 2,228 have APC pricing data. NU publications in other journals are not included in financial estimates.'},
        {'Parameter': 'TA file limitation',             'Value': 'TA files contain eISSN only. Journals with print-only ISSNs may be missed.'},
    ])
    return {'logic': logic, 'meta': meta}


fin_summary   = _build_financial_summary(nlm_df, pmc_df)
apc_breakdown = _build_apc_breakdown(nlm_df, pmc_df)
apc_logic     = _build_apc_logic_table(nu_years, nu_year_min, nu_year_max)

print(f"Financial summary rows: {len(fin_summary):,}")
print(f"APC breakdown rows    : {len(apc_breakdown):,}")
print("\n--- Financial Summary ---")
print(fin_summary[['source','apc_liability','journal_count',
                    'est_annual_apc_at_2025_price']].to_string(index=False))


Financial summary rows: 14
APC breakdown rows    : 2,184

--- Financial Summary ---
     source                             apc_liability  journal_count  est_annual_apc_at_2025_price
NLM-matched                    APC required (gold OA)          247.0                  4.152441e+06
NLM-matched Likely APC required (hybrid, PMC embargo)          149.0                  2.414592e+06
NLM-matched  Likely APC required (hybrid, not in PMC)          542.0                  7.032382e+06
NLM-matched                      No APC (diamond/S2O)           16.0                  0.000000e+00
NLM-matched                            TA waived ($0)          427.0                  0.000000e+00
NLM-matched                  Unknown (gold (diamond))            9.0                  0.000000e+00
NLM-matched        Unknown (no OA status in NIH list)          440.0                  0.000000e+00
   PMC-only                    APC required (gold OA)          194.0                  8.261960e+05
   PMC-only Likely APC re

## 17. Export to Excel

In [19]:
output_path = OUTPUT_DIR / "Galter_NLM_PMC_NIH_NU_journal_comparison_REVISED.xlsx"

def drop_cat(df):
    return df.drop(columns=['category'], errors='ignore')

sheets = [
    ('Summary',            summary),
    ('NLM+PMC+NIH',        drop_cat(df_nlm_pmc_nih)),
    ('NLM+PMC_notNIH',     drop_cat(df_nlm_pmc)),
    ('NLM+NIH_notPMC',     drop_cat(df_nlm_nih)),
    ('NLM_Only',           drop_cat(df_nlm_only)),
    ('PMC+NIH_notNLM',     drop_cat(df_pmc_nih)),
    ('PMC_Only',           drop_cat(df_pmc_only)),
    ('NIH_Only',           drop_cat(df_nih_only)),
    ('Northwestern_TA_NLM',  drop_cat(df_ta_nlm)),
    ('Northwestern_TA_PMC',  drop_cat(df_ta_pmc)),
    ('Northwestern_Pubs_NLM',drop_cat(df_nu_nlm)),
    ('Northwestern_Pubs_PMC',drop_cat(df_nu_pmc)),
    ('APC_Financial_Summary',fin_summary),
    ('APC_Journal_Breakdown',apc_breakdown),
    ('APC_Logic_Guide',    apc_logic),   # dict — handled separately below
]

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sheet_name, df in sheets:
        if isinstance(df, dict):
            # APC logic guide: two tables in one sheet
            df['logic'].to_excel(writer, sheet_name=sheet_name, index=False, startrow=0)
            start_row = len(df['logic']) + 3
            df['meta'].to_excel(writer, sheet_name=sheet_name, index=False, startrow=start_row)
            ws = writer.sheets[sheet_name]
        else:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            ws = writer.sheets[sheet_name]

        # Auto-fit column widths (approximate)
        for col_cells in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0)
                for cell in col_cells
            )
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 55)

print(f"Workbook written: {output_path}")
for sheet_name, df in sheets:
    n = len(df['logic']) + len(df['meta']) if isinstance(df, dict) else len(df)
    print(f"  {sheet_name:<28}: {n:>5,} rows")


Workbook written: output\Galter_NLM_PMC_NIH_NU_journal_comparison_REVISED.xlsx
  Summary                     :    17 rows
  NLM+PMC+NIH                 :   541 rows
  NLM+PMC_notNIH              :   854 rows
  NLM+NIH_notPMC              : 1,075 rows
  NLM_Only                    : 2,757 rows
  PMC+NIH_notNLM              :   407 rows
  PMC_Only                    : 2,567 rows
  NIH_Only                    :   232 rows
  Northwestern_TA_NLM         : 1,097 rows
  Northwestern_TA_PMC         :   220 rows
  Northwestern_Pubs_NLM       : 1,830 rows
  Northwestern_Pubs_PMC       :   354 rows
  APC_Financial_Summary       :    14 rows
  APC_Journal_Breakdown       : 2,184 rows
  APC_Logic_Guide             :    17 rows


## Notes, Caveats, and Reproducibility

### Refreshing data caches
```python
pmc_df = fetch_pmc_journals(use_cache=False)
nlm_df = fetch_nlm_journals(NLM_QUERY, use_cache=False)
```
Local files (NIH, TA, NU) are read fresh every run. Caches should be refreshed at least annually, or whenever the NLM query scope changes.

### NLM query scope
| Query | Approximate scope |
|---|---|
| `currentlyindexed[All]` | Currently indexed in MEDLINE (~5,200) |
| `ncbijournals[All]` | Broader NLM journal collection (~60,000) |

### APC cost columns reference

**Primary (use for planning):**
| Column | Formula |
|---|---|
| `est_annual_apc_at_2025_price` | `(nu_pub_count ÷ nu_years) × nih_apc_2025_usd` |
| `est_annual_apc_2k_cap` | Same with per-article cap at $2,000 |
| `est_annual_apc_3k_cap` | Same with per-article cap at $3,000 |
| `est_annual_apc_6k_cap` | Same with per-article cap at $6,000 |

**Secondary (reference/comparison only):**
| Column | Formula |
|---|---|
| `hypothetical_total_apc_at_2025_price` | `nu_pub_count × nih_apc_2025_usd` |

### Source file inventory
| Variable | File | Temporal scope |
|---|---|---|
| `NIH_FILE` | `NIH2025_2228TopJournals.csv` | Jan–Jul 2025 |
| `WILEY_FILE` | `2025_08-25 Wiley Hybrid and OA Journals.csv` | Current as of Aug 2025 |
| `SN_FILE` | `2025_BTAA_Springer_Nature_hybrid_journals.csv` | Current as of date in filename |
| `NU_FILE` | Northwestern NIH Reporter pubs | FY2020–present |

### Caveats for policy audiences
> The financial estimates in this analysis are **planning tools**, not historical accounting. They answer the question: *"If our researchers publish at the same average rate going forward, what would we expect to pay in APCs at 2025 list prices?"* They do not reflect actual expenditure, negotiated discounts, or individual author decisions. The estimates cover only journals in the NIH top 2,228 list; the uncovered portion of NU publications is quantified in the Validation section and should be disclosed when presenting totals.
